# RSNA Knee Abnormality Detection: V3 (slim)

2.5D EfficientNet-B0 MIL over 3 planes × 24 slices, trained on LLM-read report labels (all 12 findings, 4,407 studies; 58 gold labels override).

**Run all.** `RUN_MODE = "train"` builds whatever is missing, trains `FOLDS_TO_TRAIN` and predicts. `RUN_MODE = "submit"` only predicts from `checkpoints/`, and Kaggle's hidden rerun selects it automatically.

| Mode | Kaggle inputs | Settings |
|---|---|---|
| train | competition · output of `arnavkumarmd24/rsna-knee-abnormality-detection-v2` (study cache) · `arnavkumarmd24/rsna-knee-labels-v2` | GPU, Internet ON (ImageNet weights) |
| submit | competition · output(s) of the training run(s) (holding `checkpoints/`) | GPU, Internet OFF |

Finished `(config, fold)` pairs are reused. The experiment id hashes the config, labels, folds and cache, so changing any of them trains a new experiment. The previous 75-cell version (diagnostics, NLI labelling) is in `Past Version/rsna-knee-abnormality-detection-v2.before-slim.ipynb`.

In [ ]:
# Configuration, inputs, data
import os
import gc
import json
import time
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

COMPETITION = "rsna-knee-abnormality-detection"
ON_KAGGLE = Path("/kaggle/input").exists()

# "train": build what is missing, train FOLDS_TO_TRAIN, predict.
# "submit": predict only, from checkpoints/ (forced in Kaggle's hidden rerun).
RUN_MODE = "train"
if os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    RUN_MODE = "submit"

FOLDS_TO_TRAIN = [0]

if ON_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions") / COMPETITION
    if not DATA_DIR.exists():
        DATA_DIR = Path("/kaggle/input") / COMPETITION
    WORK_DIR = Path("/kaggle/working")
else:
    # Local runs need the CSVs (and any DICOMs to decode) already in DATA_DIR.
    DATA_DIR = Path("data") / COMPETITION
    WORK_DIR = Path("working")

CACHE_V2_DIR = WORK_DIR / "mri_cache_v2"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
OOF_DIR = WORK_DIR / "oof"
for path in (CACHE_V2_DIR / "studies", CHECKPOINT_DIR, OOF_DIR):
    path.mkdir(parents=True, exist_ok=True)


def input_roots():
    """Attached inputs other than the competition data."""
    if not ON_KAGGLE:
        return [Path("data")]
    return [r for r in sorted(Path("/kaggle/input").iterdir()) if r.name not in ("competitions", COMPETITION)]


def find_in_inputs(name, kind="file"):
    """Paths called `name` anywhere below an input root (Kaggle nests outputs at varying depths); skips the DICOM tree."""
    hits = []
    for root in input_roots():
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = sorted(d for d in dirnames if d != COMPETITION)
            if name in (dirnames if kind == "dir" else filenames):
                hits.append(Path(dirpath) / name)
    return hits


# A study is read from the first directory holding it; CACHE_V2_DIR (where
# missing studies are built) is searched last.
CACHE_READ_DIRS = [CACHE_V2_DIR / "studies"]

if ON_KAGGLE:
    for studies in find_in_inputs("studies", kind="dir"):
        if next(studies.glob("*.npz"), None) is not None:
            CACHE_READ_DIRS.insert(-1, studies)
    # Earlier runs' checkpoints/ and oof/ are copied in (existing files kept),
    # so finished folds are reused and submit mode finds its models.
    for name in ("checkpoints", "oof"):
        for src in find_in_inputs(name, kind="dir"):
            for f in src.rglob("*"):
                dst = WORK_DIR / name / f.relative_to(src)
                if f.is_file() and not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(f, dst)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
PLANES = ["Sagittal", "Coronal", "Axial"]
N_TARGETS, N_PLANES = len(TARGETS), len(PLANES)

# Preprocessing; changing these needs a new CACHE_V2_DIR (full cache rebuild).
N_SLICES = 24
IMG_SIZE = 256
CENTRAL_KEEP = 0.80

BATCH_SIZE = 4
NUM_WORKERS = 0 if os.name == "nt" else 4    # Windows Jupyter cannot spawn workers for notebook-defined Datasets
EPOCHS = 12
TRAIN_HOURS = 9.0                            # training stops after the epoch that would cross this (Kaggle kills sessions at 12 h)
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-2
WARMUP_EPOCHS = 1
GRAD_CLIP = 1.0
AMP_ENABLED = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")


from IPython.core.magic import register_cell_magic


@register_cell_magic
def stage(line, cell):
    """%%stage train: run the cell only when RUN_MODE == "train"."""
    if RUN_MODE != line.strip():
        print(f"Skipped: {line.strip()} stage (RUN_MODE={RUN_MODE!r})")
        return
    get_ipython().run_cell(cell).raise_error()


gpu = f" x{torch.cuda.device_count()} {torch.cuda.get_device_name(0)}" if DEVICE.type == "cuda" else ""
print(f"RUN_MODE={RUN_MODE} | {'Kaggle' if ON_KAGGLE else 'local'} | torch {torch.__version__} | {DEVICE}{gpu}")
print(f"train {train.shape} | test {test.shape} | cache dirs {[str(p) for p in CACHE_READ_DIRS]}")

In [ ]:
# Study preprocessing: DICOM → 3 planes of uint8 [<=24, 256, 256].
# The training cache was built with exactly this code; test studies go through it too.
import cv2
import pydicom


def get_slice_position(ds):
    """Slice position along the stacking direction (IPP projected on the slice normal), else InstanceNumber."""
    try:
        orientation = np.asarray(ds.ImageOrientationPatient, dtype=np.float64)
        normal = np.cross(orientation[:3], orientation[3:])
        return float(np.dot(np.asarray(ds.ImagePositionPatient, dtype=np.float64), normal))
    except Exception:
        return float(getattr(ds, "InstanceNumber", 0))


def get_ordered_dicom_files(series_path):
    """DICOM files of a series sorted by anatomical position (headers only)."""
    records = []
    for file_path in Path(series_path).iterdir():
        if not file_path.is_file():
            continue
        try:
            ds = pydicom.dcmread(str(file_path), stop_before_pixels=True, force=True)
            records.append((file_path, get_slice_position(ds), getattr(ds, "InstanceNumber", None)))
        except Exception:
            continue
    if not records:
        return []
    if all(np.isfinite(r[1]) for r in records):
        records.sort(key=lambda r: r[1])
    else:
        # Incomplete geometry: fall back to InstanceNumber.
        records.sort(key=lambda r: (r[2] is None, r[2] if r[2] is not None else 0))
    return [r[0] for r in records]


def select_slice_indices(n_slices, n_target=N_SLICES, central_keep=CENTRAL_KEEP):
    """Uniformly sample n_target slices from the central `central_keep` slab (all slices if fewer)."""
    if n_slices <= 0:
        return np.array([], dtype=np.int64)
    if n_slices <= n_target:
        return np.arange(n_slices, dtype=np.int64)
    slab_size = min(max(n_target, int(round(n_slices * central_keep))), n_slices)
    start = (n_slices - slab_size) // 2
    slab = np.arange(start, start + slab_size, dtype=np.int64)
    picked = slab[np.round(np.linspace(0, len(slab) - 1, n_target)).astype(np.int64)]
    return np.unique(picked)


def load_selected_slices(ordered_files, selected_indices, img_size=IMG_SIZE):
    """Decode slices, apply rescale, clip to the SERIES-level 1–99th percentile, resize → uint8."""
    images = []
    for idx in selected_indices:
        ds = pydicom.dcmread(str(ordered_files[int(idx)]), force=True)
        image = ds.pixel_array.astype(np.float32)
        image = image * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
        images.append(image)
    if not images:
        raise ValueError("No slices were decoded.")
    volume = np.stack(images, axis=0).astype(np.float32)
    low, high = np.percentile(volume, 1.0), np.percentile(volume, 99.0)
    if high <= low:
        high = low + 1.0
    volume = (np.clip(volume, low, high) - low) / (high - low)
    resized = np.empty((volume.shape[0], img_size, img_size), dtype=np.uint8)
    for i in range(volume.shape[0]):
        resized[i] = (
            cv2.resize(volume[i], (img_size, img_size), interpolation=cv2.INTER_AREA) * 255.0
        ).round().clip(0, 255).astype(np.uint8)
    return resized


def select_best_series_for_study(study_series_df, planes=PLANES):
    """Per plane: fluid-sensitive + fat-suppressed > fluid-sensitive > any; ties by SeriesInstanceUID."""
    selected, present_mask = {}, {}
    for plane in planes:
        candidates = study_series_df[study_series_df["Anatomical_Plane"] == plane]
        if candidates.empty:
            selected[plane], present_mask[plane] = None, 0
            continue
        fluid = pd.to_numeric(candidates["Fluid_Sensitive"], errors="coerce").fillna(0).astype(int)
        fat = pd.to_numeric(candidates["Fat_Suppression"], errors="coerce").fillna(0).astype(int)
        ranked = candidates.assign(_priority=np.where((fluid == 1) & (fat == 1), 3, np.where(fluid == 1, 2, 1)))
        ranked = ranked.sort_values(["_priority", "SeriesInstanceUID"], ascending=[False, True], kind="stable")
        selected[plane], present_mask[plane] = ranked.iloc[0]["SeriesInstanceUID"], 1
    return selected, present_mask


def preprocess_study_v2(study_id, train_series_df=train_series, img_size=IMG_SIZE, n_slices=N_SLICES,
                        central_keep=CENTRAL_KEEP, folder="train_series"):
    """One study → {"volumes": {plane: uint8 [n, img, img] or None}, "present_mask", "series_ids"}."""
    series_dir = DATA_DIR / folder
    study_df = train_series_df[train_series_df["StudyInstanceUID"] == study_id]
    if study_df.empty:
        raise ValueError(f"No series found for study {study_id}")
    selected_series, present_mask = select_best_series_for_study(study_df, planes=PLANES)
    volumes = {}
    for plane in PLANES:
        series_id = selected_series[plane]
        if series_id is None:
            volumes[plane] = None
            continue
        ordered_files = get_ordered_dicom_files(series_dir / str(study_id) / str(series_id))
        if not ordered_files:
            raise RuntimeError(f"No readable DICOM files: {plane} {series_id}")
        indices = select_slice_indices(len(ordered_files), n_target=n_slices, central_keep=central_keep)
        volume = load_selected_slices(ordered_files, indices, img_size=img_size)
        if volume.ndim != 3 or volume.shape[1:] != (img_size, img_size):
            raise RuntimeError(f"Unexpected volume shape {volume.shape}: {plane} {series_id}")
        volumes[plane] = volume
    return {"study_id": str(study_id), "volumes": volumes, "present_mask": present_mask, "series_ids": selected_series}


def resample_volume_to_24(volume, target_n=N_SLICES):
    """Linear interpolation along the slice axis only: uint8 [n, H, W] → uint8 [target_n, H, W]."""
    volume = np.asarray(volume)
    n, h, w = volume.shape
    if (h, w) != (IMG_SIZE, IMG_SIZE) or n <= 0:
        raise ValueError(f"Cannot resample a volume of shape {volume.shape}")
    if n == target_n:
        return volume.copy()
    src = np.linspace(0.0, 1.0, n, dtype=np.float32)
    dst = np.linspace(0.0, 1.0, target_n, dtype=np.float32)
    right = np.clip(np.searchsorted(src, dst, side="left"), 1, n - 1)
    left = right - 1
    alpha = ((dst - src[left]) / np.maximum(src[right] - src[left], 1e-8))[:, None, None]
    out = volume[left].astype(np.float32) * (1.0 - alpha) + volume[right].astype(np.float32) * alpha
    return np.clip(np.rint(out), 0, 255).astype(np.uint8)

In [ ]:
%%stage train
# Study cache: build only the studies no cache directory holds yet.
from concurrent.futures import ThreadPoolExecutor, as_completed

cv2.setNumThreads(1)

# Everything that changes a cached study's pixels. A cache recorded with
# other settings is refused rather than mixed with new studies.
CACHE_CONFIG = {
    "cache_version": 2,
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "central_keep": CENTRAL_KEEP,
    "dtype": "uint8",
    "series_priority": "fluid+fatsat > fluid > any, then SeriesInstanceUID",
    "slice_order": "ImagePositionPatient projected on slice normal",
    "normalization": "series-level p1-p99 clip to [0,1], INTER_AREA resize, x255",
    "planes": PLANES,
}
for studies_dir in CACHE_READ_DIRS:
    config_path = studies_dir.parent / "cache_config.json"
    if config_path.exists() and json.loads(config_path.read_text()) != CACHE_CONFIG:
        raise RuntimeError(f"{studies_dir} was built with other preprocessing; detach it or set a new CACHE_V2_DIR.")
(CACHE_V2_DIR / "cache_config.json").write_text(json.dumps(CACHE_CONFIG, indent=2))


def cache_file_map():
    paths = {}
    for studies_dir in CACHE_READ_DIRS:
        for path in studies_dir.glob("*.npz"):
            paths.setdefault(path.stem, path)
    return paths


def cache_one_study(study_id):
    processed = preprocess_study_v2(study_id, train_series_df=train_series)
    arrays = {
        "study_id": np.array(study_id),
        "present_mask": np.array([processed["present_mask"].get(p, 0) for p in PLANES], dtype=np.uint8),
    }
    for plane in PLANES:
        if processed["volumes"][plane] is not None:
            arrays[f"{plane}_volume"] = processed["volumes"][plane]
    out = CACHE_V2_DIR / "studies" / f"{study_id}.npz"
    partial = out.with_suffix(".partial")      # renamed when complete: no half-written .npz
    with open(partial, "wb") as f:
        np.savez_compressed(f, **arrays)
    partial.replace(out)


pending = sorted(set(train_series["StudyInstanceUID"].astype(str)) - set(cache_file_map()))
print(f"Cache: {len(cache_file_map()):,} studies found, {len(pending):,} to build")

cache_failures = {}
if pending:
    t0 = time.perf_counter()
    with ThreadPoolExecutor(min(4, os.cpu_count() or 1)) as pool:
        futures = {pool.submit(cache_one_study, uid): uid for uid in pending}
        for i, future in enumerate(as_completed(futures), 1):
            try:
                future.result()
            except Exception as e:
                cache_failures[futures[future]] = repr(e)
            if i % 100 == 0 or i == len(pending):
                print(f"  {i:,}/{len(pending):,} built, {len(cache_failures)} failed, {(time.perf_counter() - t0) / 60:.1f} min")
    del futures
    # Return the decode threads' freed heap to the OS before DataLoader workers fork (they would copy it page by page).
    if os.name == "posix":
        gc.collect()
        __import__("ctypes").CDLL("libc.so.6").malloc_trim(0)

uid_to_cache_path = {uid: str(path) for uid, path in cache_file_map().items()}
missing = set(train["StudyInstanceUID"].astype(str)) - set(uid_to_cache_path)
assert not missing, f"{len(missing)} training studies have no cache file, e.g. {sorted(missing)[:3]}; failures {list(cache_failures.items())[:3]}"
cache_sizes = pd.Series({uid: os.path.getsize(p) for uid, p in uid_to_cache_path.items()}).sort_index()
print(f"✓ {len(uid_to_cache_path):,} cached studies, {cache_sizes.sum() / 1024 ** 3:.1f} GB")

In [ ]:
%%stage train
# Labels: LLM-read report labels for every study (soft, 0–1); the 58 gold studies keep their true labels.
# labels_llm_soft.parquet = mean of two public CC0 sets (flight0234 v4hybrid, riadmohamed42 JEV HYBRID),
# chosen by agreement with the 58 gold studies (0.903 macro AUC vs 0.893 for the best single set).
from sklearn.metrics import roc_auc_score

labels_file = find_in_inputs("labels_llm_soft.parquet")
assert labels_file, "Attach the dataset arnavkumarmd24/rsna-knee-labels-v2 (labels_llm_soft.parquet)"
soft = pd.read_parquet(labels_file[0]).set_index("StudyInstanceUID")[TARGETS].astype(np.float32)

train_study_ids = train["StudyInstanceUID"].astype(str).to_numpy()
gold_pivot = train.loc[train[TARGETS].notna().all(axis=1)].set_index("StudyInstanceUID")[TARGETS].astype(np.float32)
gold_uids = set(gold_pivot.index)

label_auc = np.mean([roc_auc_score(gold_pivot[t], soft.loc[gold_pivot.index, t]) for t in TARGETS])
print(f"LLM labels vs the 58 gold studies: macro AUC {label_auc:.4f}")
assert label_auc > 0.85, "unexpected label file"

labels = soft.reindex(train_study_ids)
labels.loc[gold_pivot.index] = gold_pivot
assert labels.notna().all().all(), "studies without labels"
train_label_matrix = labels.to_numpy(np.float32)
train_weight_matrix = np.ones_like(train_label_matrix)
print(f"✓ {len(labels):,} studies × {N_TARGETS} soft targets ({len(gold_uids)} gold)")

In [ ]:
%%stage train
# Dataset: cached study → uint8 [3 planes, 24, 256, 256] + soft labels + weights.
from torch.utils.data import Dataset


class RSNAKneeV2Dataset(Dataset):

    def __init__(self, study_ids, label_matrix, weight_matrix, cache_paths):
        self.study_ids = np.asarray(study_ids).astype(str)
        self.labels = np.asarray(label_matrix, dtype=np.float32)
        self.weights = np.asarray(weight_matrix, dtype=np.float32)
        self.cache_paths = cache_paths
        assert len(self.study_ids) == len(self.labels) == len(self.weights)
        assert ((self.labels >= 0) & (self.labels <= 1)).all() and (self.weights >= 0).all()

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, index):
        uid = self.study_ids[index]
        images = np.zeros((N_PLANES, N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)   # a missing plane stays zero
        with np.load(self.cache_paths[uid], allow_pickle=False) as data:
            assert str(data["study_id"].item()) == uid, f"cache UID mismatch for {uid}"
            for p, plane in enumerate(PLANES):
                if f"{plane}_volume" in data.files:
                    volume = data[f"{plane}_volume"]
                    images[p] = volume if volume.shape[0] == N_SLICES else resample_volume_to_24(volume)
        return {
            "images": torch.from_numpy(images),
            "labels": torch.from_numpy(self.labels[index].copy()),
            "weights": torch.from_numpy(self.weights[index].copy()),
            "uid": uid,
        }

In [ ]:
# 2.5D MIL model: shared ImageNet EfficientNet-B0 over centred 3-slice triplets,
# attention pooling over slices, learned plane embeddings, MLP fusion of the 3 planes.
import torch.nn as nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

CNN_MICRO_BATCH = 96      # images per CNN call for no-grad / checkpointed passes
DROPOUT = 0.20
# Measured: channels_last trained 8x slower on a laptop RTX 4060 (depthwise convs)
# but ran CPU inference 1.55x faster.
CHANNELS_LAST = DEVICE.type == "cpu"
# T4 (15 GB) fits batch 4 × 3 planes × 12 slices without checkpointing; 8 GB GPUs do not.
GRAD_CHECKPOINT = bool(torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory < 12 * 1024 ** 3)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32)


class EfficientNetSliceEncoder(nn.Module):

    def __init__(self, pretrained=True):
        super().__init__()
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_dim = 1280

    def forward(self, x):
        return torch.flatten(self.pool(self.features(x)), 1)


class SliceAttentionPooling(nn.Module):

    def __init__(self, embed_dim, attention_dim=256):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(embed_dim, attention_dim), nn.Tanh(), nn.Linear(attention_dim, 1))

    def forward(self, x):
        attention = torch.softmax(self.attention(x), dim=1)        # [B, S, 1]
        return torch.sum(attention * x, dim=1), attention.squeeze(-1)


class RSNAKnee2p5DMIL(nn.Module):

    def __init__(self, n_targets=12, n_planes=3, micro_batch=CNN_MICRO_BATCH, dropout=0.20, train_slices=None,
                 grad_checkpoint=False, pretrained=True, train_chunk=None, channels_last=True):
        super().__init__()
        self.n_planes = n_planes
        self.micro_batch = micro_batch
        self.train_slices = train_slices      # train on evenly spaced slices, infer on all 24
        self.grad_checkpoint = grad_checkpoint
        self.train_chunk = train_chunk        # images per CNN call while training (= BatchNorm batch)
        self.channels_last = channels_last
        self.encoder = EfficientNetSliceEncoder(pretrained=pretrained)
        self.embed_dim = self.encoder.out_dim
        self.slice_attention = SliceAttentionPooling(self.embed_dim, 256)
        self.plane_embedding = nn.Parameter(torch.zeros(n_planes, self.embed_dim))
        nn.init.normal_(self.plane_embedding, mean=0.0, std=0.02)
        fusion_dim = self.embed_dim * n_planes
        self.fusion = nn.Sequential(
            nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, 1024), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(1024, 512), nn.GELU(), nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(512, n_targets)
        # uint8 x → (x / 255 - mean) / std == x * scale + shift (non-persistent: state_dict unchanged)
        self.register_buffer("input_scale", (1.0 / (255.0 * IMAGENET_STD)).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("input_shift", (-IMAGENET_MEAN / IMAGENET_STD).view(1, 3, 1, 1), persistent=False)

    @staticmethod
    def make_triplets(volume):
        # [B, S, H, W] → [B, S, 3, H, W] = slices (i-1, i, i+1), edges replicated
        S = volume.shape[1]
        padded = torch.cat([volume[:, :1], volume, volume[:, -1:]], dim=1)
        return torch.stack([padded[:, i:i + S] for i in range(3)], dim=2)

    def _encode(self, chunk):
        layout = torch.channels_last if self.channels_last else torch.contiguous_format
        chunk = (chunk.float() * self.input_scale + self.input_shift).contiguous(memory_format=layout)
        if self.grad_checkpoint and torch.is_grad_enabled():
            # ponytail: recompute gives BatchNorm running stats a second update per step; < 12 GB GPUs only.
            return checkpoint(self.encoder, chunk, use_reentrant=False)
        return self.encoder(chunk)

    def encode_triplets(self, triplets):
        B, S, C, H, W = triplets.shape
        flat = triplets.reshape(B * S, C, H, W)
        step = (self.train_chunk or flat.shape[0]) if torch.is_grad_enabled() and not self.grad_checkpoint else self.micro_batch
        embeddings = torch.cat([self._encode(flat[i:i + step]) for i in range(0, flat.shape[0], step)], dim=0)
        return embeddings.reshape(B, S, self.embed_dim)

    def forward(self, images, return_attention=False):
        # images: [B, 3 planes, 24, 256, 256], uint8 or float in 0–255
        B, P, S, H, W = images.shape
        assert P == self.n_planes and S == N_SLICES and H == W == IMG_SIZE, images.shape
        triplets = self.make_triplets(images.reshape(B * P, S, H, W))
        if self.training and self.train_slices and self.train_slices < S:
            keep = torch.linspace(0, S - 1, self.train_slices, device=images.device).round().long()
            triplets = triplets[:, keep]
        embeddings = self.encode_triplets(triplets)
        S_used = embeddings.shape[1]
        embeddings = embeddings.reshape(B, P, S_used, self.embed_dim) + self.plane_embedding.view(1, P, 1, self.embed_dim)
        pooled, attention = self.slice_attention(embeddings.reshape(B * P, S_used, self.embed_dim))
        logits = self.classifier(self.fusion(pooled.reshape(B, P * self.embed_dim)))
        return (logits, attention.reshape(B, P, S_used)) if return_attention else logits


def load_model_state(model, state_dict):
    """Strict load into model (or its DataParallel .module); strips torch.compile / DataParallel prefixes."""
    cleaned = {}
    for key, value in state_dict.items():
        for prefix in ("_orig_mod.", "module."):
            if key.startswith(prefix):
                key = key[len(prefix):]
        cleaned[key] = value
    getattr(model, "module", model).load_state_dict(cleaned, strict=True)
    return model

In [ ]:
%%stage train
# Loss: soft-label BCE, averaged over supervised entries (weight > 0).
import torch.nn.functional as F


def weighted_bce(logits, labels, weights, eps=1e-8):
    """Returns (loss, per-target loss)."""
    bce = F.binary_cross_entropy_with_logits(logits.float(), labels.float(), reduction="none") * weights
    return bce.sum() / (weights.sum() + eps), bce.sum(0) / (weights.sum(0) + eps)


# Check: equals plain mean BCE when every weight is 1.
_z, _y = torch.randn(5, N_TARGETS), torch.rand(5, N_TARGETS)
assert torch.allclose(weighted_bce(_z, _y, torch.ones_like(_y))[0], F.binary_cross_entropy_with_logits(_z, _y), atol=1e-6)

In [ ]:
%%stage train
# Cross-validated training engine.
# Folds: 5-fold StratifiedKFold (on the number of findings ≥ 0.5), run separately for the
# 58 gold and the 4,349 weak studies. Fold k's model trains on the other folds and is scored on fold k:
#   A = macro AUC vs the LLM labels (binarized at 0.5) on fold k's ~870 weak studies (selects the epoch)
#   B = macro AUC vs gold on fold k's ~12 gold studies (reported, never tuned on; pooled over folds
#       it becomes the cross-fitted gold-58 score)
# Experiment id = hash(config, labels, folds, cache): a finished fold is reused, an interrupted one resumes.
import hashlib
import psutil
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold

BASE_TRAIN_CONFIG = {
    "model": "efficientnet_b0_2p5d_mil_v1",
    "labels": "llm_soft_v2",
    "train_slices": 12,          # 12 of 24 slices while training, all 24 at inference
    "cnn_chunk": None,           # images per CNN call while training (None: whole batch)
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS,
    "grad_clip": GRAD_CLIP,
    "dropout": DROPOUT,
    "amp": AMP_ENABLED,
    "augment": False,
    "patience": 3,               # epochs without a better A before stopping
    "seed": SEED,
}
N_CV_FOLDS = 5

all_uids = train_study_ids
is_gold = np.isin(all_uids, list(gold_uids))
study_fold = np.empty(len(all_uids), dtype=int)
strata = np.clip((train_label_matrix >= 0.5).sum(1), 0, 4)
for group in (is_gold, ~is_gold):
    rows = np.where(group)[0]
    splitter = StratifiedKFold(N_CV_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, val) in enumerate(splitter.split(rows, strata[rows])):
        study_fold[rows[val]] = k
print(pd.crosstab(study_fold, is_gold, rownames=["fold"], colnames=["gold"]).to_string())


def _sha1(*parts):
    h = hashlib.sha1()
    for part in parts:
        h.update(part if isinstance(part, bytes) else str(part).encode())
    return h.hexdigest()


DATA_FINGERPRINT = _sha1(
    all_uids.astype(str).tobytes(), train_label_matrix.tobytes(), train_weight_matrix.tobytes(),
    study_fold.tobytes(), cache_sizes.to_csv(),
)[:12]


def experiment_id(cfg):
    return _sha1(json.dumps(cfg, sort_keys=True), DATA_FINGERPRINT)[:10]


def macro_auc(y, p, w=None):
    """Mean per-target AUC (y binarized at 0.5) over targets with both classes among supervised entries."""
    per_target = {}
    for t, target in enumerate(TARGETS):
        keep = np.ones(len(y), bool) if w is None else w[:, t] > 0
        yt = (y[keep, t] >= 0.5).astype(int)
        if 0 < yt.sum() < len(yt):
            per_target[target] = roc_auc_score(yt, p[keep, t])
    return (float(np.mean(list(per_target.values()))) if per_target else float("nan")), per_target


def bootstrap_macro_auc(y, p, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    scores = [macro_auc(y[i], p[i])[0] for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.nanpercentile(scores, [2.5, 97.5])


def gpu_augment(images, generator=None):
    """[B, P, S, H, W] uint8 → float 0–255. Per (study, plane): rotation ±10°, scale 0.9–1.1, shift ±6%,
    contrast 0.8–1.2, brightness ±10%. No flips: laterality and sagittal orientation carry meaning."""
    B, P, S, H, W = images.shape
    n = B * P
    x = images.reshape(n, S, H, W).float()
    r = lambda lo, hi: torch.empty(n, device=x.device).uniform_(lo, hi, generator=generator)
    angle, scale = r(-10, 10) * (np.pi / 180), r(0.9, 1.1)
    cos, sin = torch.cos(angle) / scale, torch.sin(angle) / scale
    theta = torch.stack([torch.stack([cos, -sin, r(-0.06, 0.06)], 1), torch.stack([sin, cos, r(-0.06, 0.06)], 1)], 1)
    grid = F.affine_grid(theta, (n, S, H, W), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    x = x * r(0.8, 1.2).view(n, 1, 1, 1) + r(-25.5, 25.5).view(n, 1, 1, 1)
    return x.clamp_(0, 255).reshape(B, P, S, H, W)


def fold_data(cfg, fold):
    train_rows = np.where(study_fold != fold)[0]
    val_rows = np.where(study_fold == fold)[0]
    dataset = lambda rows: RSNAKneeV2Dataset(all_uids[rows], train_label_matrix[rows], train_weight_matrix[rows], uid_to_cache_path)
    return train_rows, val_rows, dataset(train_rows), dataset(val_rows)


def make_loader(dataset, batch_size, shuffle, seed=0, drop_last=False):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",       # workers re-fork each epoch: a T4 run died of host RAM
        prefetch_factor=2 if NUM_WORKERS > 0 else None, generator=torch.Generator().manual_seed(seed),
    )


def build_model(cfg, pretrained=True):
    model = RSNAKnee2p5DMIL(
        n_targets=N_TARGETS, n_planes=N_PLANES, micro_batch=CNN_MICRO_BATCH, dropout=cfg["dropout"],
        train_slices=cfg["train_slices"], grad_checkpoint=GRAD_CHECKPOINT, pretrained=pretrained,
        train_chunk=cfg["cnn_chunk"], channels_last=CHANNELS_LAST,
    ).to(DEVICE, memory_format=torch.channels_last if CHANNELS_LAST else torch.contiguous_format)
    # One GPU: on Kaggle T4 x2 (torch 2.11) nn.DataParallel leaked ~30 MB host RAM per step and ran 2x slower than one T4.
    return model


def unwrap(model):
    return getattr(model, "module", model)


@torch.no_grad()
def predict(model, loader, amp=True):
    model.eval()
    out = []
    for batch in loader:
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=amp and DEVICE.type == "cuda"):
            out.append(model(batch["images"].to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy()


def run_fold(cfg, fold, log_every=100):
    exp_id = experiment_id(cfg)
    tag = f"{exp_id}_fold{fold}"
    best_path, last_path = CHECKPOINT_DIR / f"{tag}_best.pt", CHECKPOINT_DIR / f"{tag}_last.pt"
    summary_path, oof_path = CHECKPOINT_DIR / f"{tag}.json", OOF_DIR / f"{tag}.csv"

    if summary_path.exists() and json.loads(summary_path.read_text()).get("complete"):
        summary = json.loads(summary_path.read_text())
        print(f"[{tag}] reused: A={summary['best_A']:.4f}  B={summary['best_B']:.4f}")
        return summary

    torch.manual_seed(cfg["seed"] + fold)
    np.random.seed(cfg["seed"] + fold)
    train_rows, val_rows, train_ds, val_ds = fold_data(cfg, fold)
    train_loader = make_loader(train_ds, cfg["batch_size"], shuffle=True, seed=cfg["seed"] + fold, drop_last=True)
    val_loader = make_loader(val_ds, cfg["batch_size"], shuffle=False)
    val_gold = is_gold[val_rows]
    val_y, val_w = train_label_matrix[val_rows], train_weight_matrix[val_rows]

    model = build_model(cfg)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    def lr_lambda(epoch):
        # Linear warm-up, then cosine decay over the remaining epochs.
        if epoch < cfg["warmup_epochs"]:
            return float(epoch + 1) / cfg["warmup_epochs"]
        progress = (epoch - cfg["warmup_epochs"]) / max(cfg["epochs"] - cfg["warmup_epochs"], 1)
        return 0.5 * (1.0 + np.cos(np.pi * min(max(progress, 0.0), 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    use_amp = cfg["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    state = {"epoch": 0, "best_score": -np.inf, "best_A": np.nan, "best_B": np.nan, "best_epoch": -1, "history": []}

    if last_path.exists():
        ckpt = torch.load(last_path, map_location="cpu", weights_only=False)
        load_model_state(model, ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        scaler.load_state_dict(ckpt["scaler"])
        state = ckpt["state"]
        print(f"[{tag}] resuming after epoch {state['epoch']}")

    print(f"[{tag}] train {len(train_rows):,} ({int(is_gold[train_rows].sum())} gold) | val {len(val_rows):,} "
          f"({int(val_gold.sum())} gold) | {len(train_loader)} steps/epoch")
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    gen = torch.Generator(device=DEVICE).manual_seed(cfg["seed"] + fold)
    fold_t0 = time.perf_counter()

    for epoch in range(state["epoch"], cfg["epochs"]):
        model.train()
        t0, data_wait = time.perf_counter(), 0.0
        loss_sum = torch.zeros((), device=DEVICE)
        t_fetch = time.perf_counter()
        for step, batch in enumerate(train_loader):
            data_wait += time.perf_counter() - t_fetch
            images = batch["images"].to(DEVICE, non_blocking=True)
            labels_b = batch["labels"].to(DEVICE, non_blocking=True)
            weights_b = batch["weights"].to(DEVICE, non_blocking=True)
            if cfg["augment"]:
                images = gpu_augment(images, gen)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
                logits = model(images)
            loss, _ = weighted_bce(logits, labels_b, weights_b)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.detach()
            if log_every and (step + 1) % log_every == 0:
                print(f"  epoch {epoch + 1} step {step + 1}/{len(train_loader)} loss {loss_sum.item() / (step + 1):.4f} "
                      f"({(time.perf_counter() - t0) / 60:.1f} min, data wait {100 * data_wait / (time.perf_counter() - t0):.0f}%, "
                      f"RAM free {psutil.virtual_memory().available / 1024 ** 3:.1f} GB)")
            t_fetch = time.perf_counter()
        scheduler.step()
        train_sec = time.perf_counter() - t0

        t1 = time.perf_counter()
        val_logits = predict(model, val_loader, amp=use_amp)
        val_prob = 1.0 / (1.0 + np.exp(-np.clip(val_logits, -30, 30)))
        A, _ = macro_auc(val_y[~val_gold], val_prob[~val_gold], val_w[~val_gold])
        B, _ = macro_auc(val_y[val_gold], val_prob[val_gold])
        val_loss = float(weighted_bce(torch.from_numpy(val_logits), torch.from_numpy(val_y), torch.from_numpy(val_w))[0])

        record = {
            "epoch": epoch + 1, "train_loss": loss_sum.item() / max(len(train_loader), 1), "val_loss": val_loss,
            "A": A, "B": B, "lr": optimizer.param_groups[0]["lr"], "train_min": train_sec / 60,
            "val_min": (time.perf_counter() - t1) / 60,
            "studies_per_s": len(train_loader) * train_loader.batch_size / train_sec,
            "data_wait_pct": 100 * data_wait / train_sec,
            "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1024 ** 3 if DEVICE.type == "cuda" else 0.0,
            "cpu_rss_gb": psutil.Process().memory_info().rss / 1024 ** 3,
        }
        state["history"].append(record)
        state["epoch"] = epoch + 1
        score = A if np.isfinite(A) else -val_loss
        improved = score > state["best_score"] + 1e-4
        print(f"[{tag}] epoch {epoch + 1}/{cfg['epochs']} | loss {record['train_loss']:.4f} | val loss {val_loss:.4f} "
              f"A {A:.4f} B {B:.4f} | {record['train_min']:.1f}+{record['val_min']:.1f} min, "
              f"{record['studies_per_s']:.1f} st/s, wait {record['data_wait_pct']:.0f}%, {record['peak_gpu_gb']:.1f} GB"
              + ("  ← best" if improved else ""))

        if improved:
            state.update(best_score=score, best_A=A, best_B=B, best_epoch=epoch + 1)
            torch.save({"model": unwrap(model).state_dict(), "config": cfg, "exp_id": exp_id, "fold": fold,
                        "epoch": epoch + 1, "A": A, "B": B, "targets": TARGETS}, best_path)
            oof = pd.DataFrame(val_prob, columns=TARGETS)
            oof.insert(0, "StudyInstanceUID", all_uids[val_rows])
            oof.insert(1, "IsGold", val_gold)
            oof.to_csv(oof_path, index=False)

        torch.save({"model": unwrap(model).state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "state": state}, last_path)

        if epoch + 1 - state["best_epoch"] >= cfg["patience"]:
            print(f"[{tag}] early stop: no better A for {cfg['patience']} epochs")
            break
        epoch_sec = time.perf_counter() - t0
        if (time.perf_counter() - fold_t0 + epoch_sec) / 3600 > TRAIN_HOURS:
            print(f"[{tag}] time budget: stopping after epoch {epoch + 1} (TRAIN_HOURS={TRAIN_HOURS}), best epoch {state['best_epoch']}")
            break

    summary = {"exp_id": exp_id, "fold": fold, "config": cfg, "complete": True, "best_A": state["best_A"],
               "best_B": state["best_B"], "best_epoch": state["best_epoch"], "history": state["history"]}
    summary_path.write_text(json.dumps(summary, indent=1, default=float))
    last_path.unlink(missing_ok=True)     # inference uses the best checkpoint
    del model, optimizer, train_loader, val_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return summary


def cross_fitted_gold(cfg):
    """Gold-58 macro AUC from the saved OOF files of every finished fold, with a bootstrap CI."""
    parts = [pd.read_csv(p) for p in sorted(OOF_DIR.glob(f"{experiment_id(cfg)}_fold*.csv"))]
    if not parts:
        return None
    oof = pd.concat(parts, ignore_index=True)
    gold = oof[oof["IsGold"]].set_index("StudyInstanceUID")
    y, p = gold_pivot.loc[gold.index, TARGETS].to_numpy(), gold[TARGETS].to_numpy()
    macro, per_target = macro_auc(y, p)
    return {"n_gold": len(gold), "folds": len(parts), "macro": macro, "ci": tuple(bootstrap_macro_auc(y, p)), "per_target": per_target}


print(f"✓ Training engine ready | data fingerprint {DATA_FINGERPRINT} | experiment {experiment_id(BASE_TRAIN_CONFIG)} "
      f"| grad checkpointing {GRAD_CHECKPOINT} | workers {NUM_WORKERS}")

In [ ]:
%%stage train
# Train: a 10-step sanity check on one batch, then FOLDS_TO_TRAIN (finished folds are reused).
import io


def level_a_sanity(cfg, steps=10):
    """Loss must fall on a fixed batch, gradients must reach every parameter, a checkpoint must round-trip."""
    t0 = time.perf_counter()
    _, _, train_ds, _ = fold_data(cfg, 0)
    batch = next(iter(make_loader(train_ds, cfg["batch_size"], shuffle=True, seed=0)))
    images, labels_b, weights_b = (batch[k].to(DEVICE) for k in ("images", "labels", "weights"))
    assert images.shape[1:] == (N_PLANES, N_SLICES, IMG_SIZE, IMG_SIZE) and images.dtype == torch.uint8, images.shape

    model = build_model(cfg)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])
    use_amp = cfg["amp"] and DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    losses, finite = [], 0
    model.train()
    for _ in range(steps):
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
            loss, _ = weighted_bce(model(images), labels_b, weights_b)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        assert all(p.grad is not None for p in model.parameters()), "parameters without gradient"
        scaler.unscale_(optimizer)
        # An early fp16 overflow step (inf norm) is normal: GradScaler skips it.
        finite += bool(torch.isfinite(torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])))
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    assert finite >= steps // 2 and losses[-1] < losses[0], f"no learning on a fixed batch: {losses}"

    model.eval()
    buffer = io.BytesIO()
    torch.save({"model": unwrap(model).state_dict()}, buffer)
    buffer.seek(0)
    restored = load_model_state(build_model(cfg, pretrained=False), torch.load(buffer, weights_only=True)["model"]).eval()
    saved, loaded = unwrap(model).state_dict(), unwrap(restored).state_dict()
    assert saved.keys() == loaded.keys() and all(torch.equal(saved[k], loaded[k]) for k in saved), "checkpoint round trip differs"
    # One eval pass as predict() runs it (fp16 autocast; fp32 convs hit "CUDA error: misaligned address" on Kaggle T4, torch 2.11).
    # Outputs are not compared: cudnn.benchmark may pick different fp16 kernels for the two models.
    with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
        assert torch.isfinite(restored(images).float()).all(), "non-finite eval output"
    peak = torch.cuda.max_memory_allocated() / 1024 ** 3 if DEVICE.type == "cuda" else 0.0
    print(f"✓ Sanity check in {time.perf_counter() - t0:.0f}s: batch {tuple(images.shape)}, loss {losses[0]:.4f} → {losses[-1]:.4f}, "
          f"checkpoint round trip exact, peak {peak:.1f} GB")
    del model, restored, optimizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


level_a_sanity(BASE_TRAIN_CONFIG)

FINAL_EXPERIMENT = experiment_id(BASE_TRAIN_CONFIG)
fold_summaries = [run_fold(BASE_TRAIN_CONFIG, fold) for fold in FOLDS_TO_TRAIN]

for s in fold_summaries:
    print(f"\nFold {s['fold']} (best epoch {s['best_epoch']}):")
    print(pd.DataFrame(s["history"])[["epoch", "train_loss", "val_loss", "A", "B", "train_min", "studies_per_s", "data_wait_pct", "peak_gpu_gb"]].round(4).to_string(index=False))

gold_cv = cross_fitted_gold(BASE_TRAIN_CONFIG)
if gold_cv:
    print(f"\nCross-fitted gold macro AUC ({gold_cv['n_gold']} gold studies, {gold_cv['folds']} fold(s)): "
          f"{gold_cv['macro']:.4f} (95% bootstrap CI {gold_cv['ci'][0]:.3f}–{gold_cv['ci'][1]:.3f})")
    print(pd.Series(gold_cv["per_target"]).round(4).to_string())

In [ ]:
# Test inference + validated submission.csv (runs in both modes).
# Test studies go through the training preprocessing; a thread pool decodes DICOMs while the GPU
# predicts; the fold models of one experiment are averaged. A study that fails to decode gets the
# gold prevalence, and a missing plane is zero-filled; both are reported.
from concurrent.futures import ThreadPoolExecutor

INFER_THREADS = 4

by_experiment = {}
for path in sorted(CHECKPOINT_DIR.glob("*_fold*_best.pt")):
    by_experiment.setdefault(path.name.split("_fold")[0], []).append(path)
if not by_experiment:
    raise FileNotFoundError(f"No *_best.pt in {CHECKPOINT_DIR}: train first, or attach a training run's output.")

# The experiment trained above, else the one with the most fold checkpoints (newest wins ties).
INFER_EXPERIMENT = globals().get("FINAL_EXPERIMENT")
if INFER_EXPERIMENT not in by_experiment:
    INFER_EXPERIMENT = max(by_experiment, key=lambda e: (len(by_experiment[e]), max(p.stat().st_mtime for p in by_experiment[e])))

models = []
for path in by_experiment[INFER_EXPERIMENT]:
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    assert list(ckpt["targets"]) == TARGETS, f"{path.name}: target order differs"
    cfg = ckpt["config"]
    model = RSNAKnee2p5DMIL(n_targets=N_TARGETS, n_planes=N_PLANES, micro_batch=CNN_MICRO_BATCH, dropout=cfg["dropout"],
                            train_slices=cfg["train_slices"], pretrained=False, channels_last=CHANNELS_LAST)
    load_model_state(model, ckpt["model"])
    models.append(model.to(DEVICE, memory_format=torch.channels_last if CHANNELS_LAST else torch.contiguous_format).eval())
    print(f"Loaded {path.name}: fold {ckpt['fold']}, epoch {ckpt['epoch']}, A={ckpt['A']:.4f}, B={ckpt['B']:.4f}")

prior = train.loc[train[TARGETS].notna().all(axis=1), TARGETS].astype(float).mean().to_numpy()
decode_sec, model_sec = [], []


def load_test_study(study_id):
    t = time.perf_counter()
    processed = preprocess_study_v2(study_id, train_series_df=test_series, folder="test_series")
    images = np.zeros((N_PLANES, N_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    missing = []
    for p, plane in enumerate(PLANES):
        volume = processed["volumes"][plane]
        if volume is None:
            missing.append(plane)
        else:
            images[p] = volume if volume.shape[0] == N_SLICES else resample_volume_to_24(volume)
    decode_sec.append(time.perf_counter() - t)
    return images, missing


test_ids = test["StudyInstanceUID"].astype(str).tolist()
predictions = np.zeros((len(test_ids), N_TARGETS))
failures, missing_planes = {}, {}
window = 2 * INFER_THREADS
t0 = time.perf_counter()

with ThreadPoolExecutor(INFER_THREADS) as pool:
    futures = {i: pool.submit(load_test_study, test_ids[i]) for i in range(min(window, len(test_ids)))}
    for i, uid in enumerate(test_ids):
        if i + window < len(test_ids):
            futures[i + window] = pool.submit(load_test_study, test_ids[i + window])
        try:
            images, missing = futures.pop(i).result()
            if missing:
                missing_planes[uid] = missing
            t_model = time.perf_counter()
            x = torch.from_numpy(images).unsqueeze(0).to(DEVICE)
            with torch.no_grad(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                predictions[i] = torch.stack([torch.sigmoid(m(x).float()) for m in models]).mean(0)[0].cpu().numpy()
            model_sec.append(time.perf_counter() - t_model)
        except Exception as e:
            failures[uid] = repr(e)
            predictions[i] = prior
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{len(test_ids)} studies, {(time.perf_counter() - t0) / (i + 1):.2f} s/study")
infer_sec = time.perf_counter() - t0


def validate_submission(sub):
    assert list(sub.columns) == list(sample_submission.columns), "columns differ from sample_submission"
    assert sub["StudyInstanceUID"].astype(str).tolist() == test["StudyInstanceUID"].astype(str).tolist(), "ids / order differ from test.csv"
    values = sub[TARGETS].to_numpy(dtype=float)
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all(), "predictions not finite in [0, 1]"


submission = pd.DataFrame(predictions, columns=TARGETS)
submission.insert(0, "StudyInstanceUID", test_ids)
validate_submission(submission)
submission.to_csv(WORK_DIR / "submission.csv", index=False)
validate_submission(pd.read_csv(WORK_DIR / "submission.csv"))

print(f"\nSubmission: experiment {INFER_EXPERIMENT} ({len(models)} fold model(s)), {len(test_ids)} studies in {infer_sec:.0f}s "
      f"({infer_sec / max(len(test_ids), 1):.2f} s/study)")
if model_sec:
    # Decoding runs on INFER_THREADS threads while the model predicts: the slower of the two sets the pace.
    steady = max(np.median(decode_sec) / INFER_THREADS, np.median(model_sec[1:] or model_sec))
    print(f"  decode median {np.median(decode_sec):.2f} s/study/thread | model median {np.median(model_sec[1:] or model_sec):.2f} s/study "
          f"| projection ~{(model_sec[0] + 1300 * steady) / 60:.0f} min for 1,300 studies")
print(f"  decode failures {len(failures)} {list(failures.items())[:3]} | missing planes {len(missing_planes)} {list(missing_planes.items())[:3]}")
print(f"✓ {WORK_DIR / 'submission.csv'} validated (columns, ids/order, finite values in [0, 1])")